# App-20b : Benchmark compare des solveurs Sudoku (jumeau C#)

> **Twin C#** de [`App-20-SudokuBenchmark-Python`](App-20-SudokuBenchmark-Python.ipynb) (Python stdlib : `time`, `random`, `copy`).
> Marathon **.NET / Python** (#4956), volet **Search / Applications / CSP**.

**Navigation** : [<< App-20 Python](App-20-SudokuBenchmark-Python.ipynb) | [Index](../../README.md) | [App-7 Wordle >>](App-7-Wordle.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Implementer from-scratch** quatre solveurs Sudoku d'complexite croissante : backtracking naif, backtracking + MRV, AC-3 + backtracking, Dancing Links (Knuth 2000).
2. **Mesurer** le cout exact (nombre d'appels recursifs, temps) de chaque approche sur un banc de trois grilles de difficulte croissante.
3. **Comparer** l'impact des heuristiques (MRV) et de la propagation de contraintes (AC-3) sur la taille de l'arbre de recherche.
4. **Apprecier** pourquoi Dancing Links (formulation exact-cover) est asymptotiquement superieur au backtracking classical.

Le code est entierement reecrit en C# (.NET 9, 0 NuGet), en miroir de la version Python qui n'utilisait que la stdlib. Le critere de parite = le **nombre d'appels recursifs** (deterministe, independant du runtime) : pour un meme solveur et une meme grille, le compte Python et C# doit concorder exactement. Le temps, lui, varie avec le runtime (C# typiquement plus rapide) et n'est pas un critere de parite.


In [1]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;

const int SEED = 42;
const double TIMEOUT_S = 60.0;
string[] DIFFICULTIES = { "Easy", "Medium", "Hard" };

Console.WriteLine($"Setup OK. Seed={SEED}, TIMEOUT={TIMEOUT_S}s");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Setup OK. Seed=42, TIMEOUT=60s


---
## 1. Banc de test -- trois grilles (Easy / Medium / Hard)

Trois grilles 9x9 representative des niveaux de difficulte (`0` = case vide). Elles sont **strictement identiques** au notebook Python, condition necessaire a la parite du nombre d'appels.


In [2]:
static readonly int[,] EASY_GRID = {
    {5,3,0, 0,7,0, 0,0,0},
    {6,0,0, 1,9,5, 0,0,0},
    {0,9,8, 0,0,0, 0,6,0},
    {8,0,0, 0,6,0, 0,0,3},
    {4,0,0, 8,0,3, 0,0,1},
    {7,0,0, 0,2,0, 0,0,6},
    {0,6,0, 0,0,0, 2,8,0},
    {0,0,0, 4,1,9, 0,0,5},
    {0,0,0, 0,8,0, 0,7,9},
};
static readonly int[,] MEDIUM_GRID = {
    {0,0,0, 2,6,0, 7,0,1},
    {6,8,0, 0,7,0, 0,9,0},
    {1,9,0, 0,0,4, 5,0,0},
    {8,2,0, 1,0,0, 0,4,0},
    {0,0,4, 6,0,2, 9,0,0},
    {0,5,0, 0,0,3, 0,2,8},
    {0,0,9, 3,0,0, 0,7,4},
    {0,4,0, 0,5,0, 0,3,6},
    {7,0,3, 0,1,8, 0,0,0},
};
static readonly int[,] HARD_GRID = {
    {0,0,0, 6,0,0, 4,0,0},
    {7,0,0, 0,0,3, 6,0,0},
    {0,0,0, 0,9,1, 0,8,0},
    {0,0,0, 0,0,0, 0,0,0},
    {0,5,0, 1,8,0, 0,0,3},
    {0,0,0, 3,0,6, 0,4,5},
    {0,4,0, 2,0,0, 0,6,0},
    {9,0,3, 0,0,0, 0,0,0},
    {0,2,0, 0,0,0, 1,0,0},
};

static readonly Dictionary<string,int[,]> GRIDS = new()
{
    {"Easy", EASY_GRID}, {"Medium", MEDIUM_GRID}, {"Hard", HARD_GRID}
};

// Copie profonde d'une grille (les solveurs muent la grille en place).
static int[,] CopyGrid(int[,] src)
{
    var dst = new int[9,9];
    for (int r=0;r<9;r++) for (int c=0;c<9;c++) dst[r,c]=src[r,c];
    return dst;
}

foreach (var (name, grid) in GRIDS)
{
    int givens = 0;
    for (int r=0;r<9;r++) for (int c=0;c<9;c++) if (grid[r,c]!=0) givens++;
    Console.WriteLine($"{name,-6} : {givens,2} indices donnes, {81-givens,2} cases vides");
}


Easy   : 30 indices donnes, 51 cases vides


Medium : 36 indices donnes, 45 cases vides


Hard   : 23 indices donnes, 58 cases vides


---
## 2. Solveur 1 -- backtracking naif

Le solveur le plus simple : trouver la premiere case vide (ordre ligne-par-ligne, colonne-par-colonne), essayer les valeurs 1..9 dans l'ordre, verifier la validite (ligne + colonne + carre 3x3), et recurser. Aucune heuristique : l'arbre de recherche explose rapidement sur les grilles difficiles.


In [3]:
// --- Solveur 1 : backtracking naif ---
static (int r, int c)? FindEmptyNaive(int[,] g)
{
    for (int r=0;r<9;r++) for (int c=0;c<9;c++)
        if (g[r,c]==0) return (r,c);
    return null;
}

static bool IsValidNaive(int[,] g, int row, int col, int val)
{
    for (int i=0;i<9;i++) { if (g[row,i]==val) return false; if (g[i,col]==val) return false; }
    int br=3*(row/3), bc=3*(col/3);
    for (int r=br;r<br+3;r++) for (int c=bc;c<bc+3;c++) if (g[r,c]==val) return false;
    return true;
}

static bool SolveBacktracking(int[,] g, ref int calls)
{
    calls++;
    var empty = FindEmptyNaive(g);
    if (empty is null) return true;
    int row=empty.Value.r, col=empty.Value.c;
    for (int val=1; val<=9; val++)
    {
        if (IsValidNaive(g, row, col, val))
        {
            g[row,col]=val;
            if (SolveBacktracking(g, ref calls)) return true;
            g[row,col]=0;
        }
    }
    return false;
}

static (bool ok, double ms, int calls) RunBacktracking(int[,] grid)
{
    var g = CopyGrid(grid);
    int calls=0;
    var sw = Stopwatch.StartNew();
    bool success = SolveBacktracking(g, ref calls);
    sw.Stop();
    return (success, sw.Elapsed.TotalMilliseconds, calls);
}

var (ok1, ms1, calls1) = RunBacktracking(EASY_GRID);
Console.WriteLine($"Backtracking simple -- Easy : succes={ok1}, temps={ms1:F2} ms, appels={calls1}");


Backtracking simple -- Easy : succes=True, temps=1,68 ms, appels=4209


### Solveur 2 -- backtracking + MRV (Minimum Remaining Values)

Le meme backtracking, mais on choisit a chaque pas la case vide **avec le moins de valeurs possibles**. L'heuristique MRV reduit drastiquement le facteur de branchement : une case avec une seule valeur possible est remplie immediatement (propagation implicite), et les branches mortes sont coupees plus tot.


In [4]:
// --- Solveur 2 : backtracking + MRV ---
static (int r, int c, List<int> possibles)? FindEmptyMRV(int[,] g)
{
    (int r, int c, List<int> possibles)? best = null;
    for (int r=0;r<9;r++) for (int c=0;c<9;c++)
    {
        if (g[r,c]!=0) continue;
        var used = new HashSet<int>();
        for (int i=0;i<9;i++) { used.Add(g[r,i]); used.Add(g[i,c]); }
        int br=3*(r/3), bc=3*(c/3);
        for (int i=br;i<br+3;i++) for (int j=bc;j<bc+3;j++) used.Add(g[i,j]);
        var poss = new List<int>();
        for (int v=1;v<=9;v++) if (!used.Contains(v)) poss.Add(v);
        if (best is null || poss.Count < best.Value.possibles.Count)
        {
            best = (r, c, poss);
            if (poss.Count==0) return best; // fail-fast : case vide sans possibilite
        }
    }
    return best;
}

static bool SolveBacktrackingMRV(int[,] g, ref int calls)
{
    calls++;
    var cell = FindEmptyMRV(g);
    if (cell is null) return true;
    int row=cell.Value.r, col=cell.Value.c;
    foreach (var val in cell.Value.possibles)
    {
        g[row,col]=val;
        if (SolveBacktrackingMRV(g, ref calls)) return true;
        g[row,col]=0;
    }
    return false;
}

static (bool ok, double ms, int calls) RunBacktrackingMRV(int[,] grid)
{
    var g = CopyGrid(grid);
    int calls=0;
    var sw = Stopwatch.StartNew();
    bool success = SolveBacktrackingMRV(g, ref calls);
    sw.Stop();
    return (success, sw.Elapsed.TotalMilliseconds, calls);
}

var (ok2, ms2, calls2) = RunBacktrackingMRV(EASY_GRID);
Console.WriteLine($"Backtracking+MRV -- Easy : succes={ok2}, temps={ms2:F2} ms, appels={calls2}");


Backtracking+MRV -- Easy : succes=True, temps=1,30 ms, appels=52


### Solveur 3 -- AC-3 + backtracking

On ajoute une **propagation de contraintes** avant la recherche : l'algorithme AC-3 (Arc Consistency 3) elimine des domaines des variables les valeurs sans support dans les domaines de leurs voisins. Pour le Sudoku, deux cases liees (meme ligne/colonne/carre) sont des variables dont les valeurs doivent etre differentes. Apres propagation, le backtracking (avec MRV) travaille sur des domaines deja reduits.

Si la propagation vide un domaine, le probleme est inconsistent (infeasible) et on arrete.


In [5]:
// --- Solveur 3 : AC-3 + backtracking ---
static List<(int r,int c)> Neighbors(int r, int c)
{
    var out_ = new List<(int,int)>();
    for (int i=0;i<9;i++)
    {
        if (i!=c) out_.Add((r,i));
        if (i!=r) out_.Add((i,c));
    }
    int br=3*(r/3), bc=3*(c/3);
    for (int i=br;i<br+3;i++) for (int j=bc;j<bc+3;j++)
        if ((i,j)!=(r,c)) out_.Add((i,j));
    return out_;
}

// Elimine de xi les valeurs sans support dans xj. Retourne true si modification.
static bool Revise(HashSet<int>[] domains, int xi, int xj, List<(int,int)> cells9)
{
    bool revised=false;
    var toRemove = new List<int>();
    foreach (var x in domains[xi])
    {
        // x a un support ssi il existe y dans domains[xj] avec y != x
        bool support=false;
        foreach (var y in domains[xj]) { if (y!=x) { support=true; break; } }
        if (!support) toRemove.Add(x);
    }
    foreach (var x in toRemove) { domains[xi].Remove(x); revised=true; }
    return revised;
}

// AC-3 standard. Retourne false si un domaine devient vide (inconsistent).
static bool AC3(HashSet<int>[] domains, List<(int r,int c)> cells9)
{
    var queue = new Queue<(int xi,int xj)>();
    // voisinages precalcules par index de cellule
    var neighOf = new List<int>[81];
    for (int i=0;i<81;i++) { neighOf[i] = new List<int>(); var (r,c)=cells9[i]; foreach (var (nr,nc) in Neighbors(r,c)) neighOf[i].Add(nr*9+nc); }
    foreach (int xi in Enumerable.Range(0,81)) foreach (var xj in neighOf[xi]) queue.Enqueue((xi,xj));
    while (queue.Count>0)
    {
        var (xi,xj) = queue.Dequeue();
        if (Revise(domains, xi, xj, cells9))
        {
            if (domains[xi].Count==0) return false;
            foreach (var xk in neighOf[xi]) if (xk!=xj) queue.Enqueue((xk,xi));
        }
    }
    return true;
}

static (bool ok, double ms, int calls) RunAC3(int[,] grid)
{
    var cells9 = new List<(int,int)>();
    for (int r=0;r<9;r++) for (int c=0;c<9;c++) cells9.Add((r,c));
    var domains = new HashSet<int>[81];
    for (int r=0;r<9;r++) for (int c=0;c<9;c++)
    {
        int idx=r*9+c;
        domains[idx] = grid[r,c]!=0 ? new HashSet<int>{grid[r,c]} : new HashSet<int>{1,2,3,4,5,6,7,8,9};
    }
    int calls=0;
    var sw = Stopwatch.StartNew();
    if (!AC3(domains, cells9)) { sw.Stop(); return (false, sw.Elapsed.TotalMilliseconds, calls); }
    // Backtracking avec MRV sur les domaines reduits
    var assign = new int?[81];
    bool Backtrack()
    {
        calls++;
        int? pick=null; int bestSize=int.MaxValue;
        for (int i=0;i<81;i++)
        {
            if (assign[i].HasValue) continue;
            if (domains[i].Count<bestSize) { bestSize=domains[i].Count; pick=i; if (bestSize==0) return false; }
        }
        if (pick is null) return true; // toutes assignees
        int v = pick.Value; var (vr,vc)=cells9[v];
        foreach (var val in domains[v].OrderBy(x=>x))
        {
            bool ok=true;
            foreach (var (nr,nc) in Neighbors(vr,vc)) { if (assign[nr*9+nc]==val) { ok=false; break; } }
            if (!ok) continue;
            assign[v]=val;
            if (Backtrack()) return true;
            assign[v]=null;
        }
        return false;
    }
    bool success = Backtrack();
    sw.Stop();
    return (success, sw.Elapsed.TotalMilliseconds, calls);
}

var (ok3, ms3, calls3) = RunAC3(EASY_GRID);
Console.WriteLine($"AC-3 -- Easy : succes={ok3}, temps={ms3:F2} ms, appels={calls3}");


AC-3 -- Easy : succes=True, temps=3,83 ms, appels=82


### Solveur 4 -- Dancing Links (Knuth 2000)

La formulation la plus elegante : Sudoku comme **couverture exacte** (exact cover). On encode le probleme en 324 contraintes (colonnes) :

- 81 contraintes "la case (r,c) est remplie" (une seule valeur par case) ;
- 81 contraintes "la ligne r contient la valeur v" ;
- 81 contraintes "la colonne c contient la valeur v" ;
- 81 contraintes "le carre b contient la valeur v".

Chaque candidat (r,c,v) couvre **exactement 4 colonnes**. L'algorithme X de Knuth, implemente avec les **Dancing Links** (listes doublement chainees toroidales), resout le probleme en O(c) par nœud visite. C'est asymptotiquement le plus rapide pour la couverture exacte.


In [6]:
// --- Solveur 4 : Dancing Links (Algorithm X, Knuth 2000) ---
class DLXNode
{
    public int Row=-1, Col=-1, Size=0;
    public DLXNode Up, Down, Left, Right;
}

class DLXSolver
{
    private DLXNode head;
    private List<DLXNode> cols;
    private int[,] solution;

    public bool Solve(int[,] grid)
    {
        const int N = 324;
        head = new DLXNode();
        var h = head; h.Left=h; h.Right=h; h.Up=h; h.Down=h;
        cols = new List<DLXNode>(N);
        // creation des colonnes
        DLXNode prev = head;
        for (int i=0;i<N;i++)
        {
            var col = new DLXNode{Col=i, Size=0};
            col.Up=col; col.Down=col; col.Left=prev; prev.Right=col; prev=col;
            cols.Add(col);
        }
        prev.Right=head; head.Left=prev;
        // creation des lignes candidates (r,c,v)
        for (int r=0;r<9;r++) for (int c=0;c<9;c++)
        {
            int v = grid[r,c];
            var candidates = v!=0 ? new[]{v} : Enumerable.Range(1,9);
            foreach (var vv in candidates) AddRow(r,c,vv);
        }
        solution = new int[9,9];
        return Search(0);
    }

    private void AddRow(int r, int c, int v)
    {
        int c0 = r*9+c;            // case (r,c) remplie
        int c1 = 81 + r*9 + (v-1); // ligne r contient v
        int c2 = 162 + c*9 + (v-1);// colonne c contient v
        int b = 3*(r/3)+(c/3);
        int c3 = 243 + b*9 + (v-1);// carre b contient v
        int[] colIdx = {c0,c1,c2,c3};
        DLXNode rowPrev=null, first=null;
        foreach (var ci in colIdx)
        {
            var col = cols[ci];
            var node = new DLXNode{Row=r*100+c*10+v, Col=ci};
            // insert en bas de la colonne
            node.Down=col; node.Up=col.Up; col.Up.Down=node; col.Up=node;
            col.Size++;
            // chaine horizontale de la ligne
            if (first is null) { first=node; node.Left=node; node.Right=node; rowPrev=node; }
            else { node.Left=rowPrev; rowPrev.Right=node; node.Right=first; first.Left=node; rowPrev=node; }
        }
    }

    private void Cover(DLXNode col) { col.Right.Left=col.Left; col.Left.Right=col.Right; for (var i=col.Down;i!=col;i=i.Down) for (var j=i.Right;j!=i;j=j.Right) { j.Down.Up=j.Up; j.Up.Down=j.Down; cols[j.Col].Size--; } }
    private void Uncover(DLXNode col) { for (var i=col.Up;i!=col;i=i.Up) for (var j=i.Left;j!=i;j=j.Left) { cols[j.Col].Size++; j.Down.Up=j; j.Up.Down=j; } col.Right.Left=col; col.Left.Right=col; }

    private bool Search(int depth)
    {
        if (head.Right==head) return true; // toutes les colonnes couvertes = solution
        // choisir la colonne de plus petite taille (heuristique S)
        DLXNode col=head.Right; int best=col.Size;
        for (var c=head.Right;c!=head;c=c.Right) if (c.Size<best) { best=c.Size; col=c; }
        Cover(col);
        for (var r=col.Down;r!=col;r=r.Down)
        {
            for (var j=r.Right;j!=r;j=j.Right) Cover(cols[j.Col]);
            // decoder (r,c,v) depuis le Row tag
            int tag=r.Row; int rr=tag/100, cc=(tag/10)%10, vv=tag%10;
            solution[rr,cc]=vv;
            if (Search(depth+1)) return true;
            for (var j=r.Left;j!=r;j=j.Left) Uncover(cols[j.Col]);
        }
        Uncover(col);
        return false;
    }

    public int[,] GetSolution() => solution;
}

static (bool ok, double ms, int calls) RunDLX(int[,] grid)
{
    var g = CopyGrid(grid);
    var solver = new DLXSolver();
    var sw = Stopwatch.StartNew();
    bool success = solver.Solve(g);
    sw.Stop();
    return (success, sw.Elapsed.TotalMilliseconds, 0); // DLX n'expose pas calls directement (cf. exercice 2)
}

var (ok4, ms4, calls4) = RunDLX(EASY_GRID);
Console.WriteLine($"DLX (Knuth) -- Easy : succes={ok4}, temps={ms4:F2} ms");


DLX (Knuth) -- Easy : succes=True, temps=0,89 ms


---
## 3. Benchmark compare -- les quatre solveurs sur trois difficultes

On execute les quatre solveurs sur les trois grilles et on mesure : (a) le succes, (b) le nombre d'appels recursifs (deterministe, critere de parite), (c) le temps wall-clock (informatif, varie avec le runtime).


In [7]:
// Wrapper pour uniformiser la signature des 4 solveurs dans le benchmark.
static (bool ok, double ms, int calls) RunByName(string sname, int[,] grid) => sname switch
{
    "Backtracking simple" => RunBacktracking(grid),
    "Backtracking + MRV"  => RunBacktrackingMRV(grid),
    "AC-3 + backtracking" => RunAC3(grid),
    "DLX (Knuth)"         => RunDLX(grid),
    _ => (false, 0, 0)
};
string[] SOLVER_NAMES = { "Backtracking simple", "Backtracking + MRV", "AC-3 + backtracking", "DLX (Knuth)" };

Console.WriteLine($"{"Diff",6} | {"Solveur",-22} | {"Succes",-7} | {"Temps(ms)",10} | {"Appels",10}");
Console.WriteLine(new string('-', 65));
foreach (var diffName in DIFFICULTIES)
{
    var grid = GRIDS[diffName];
    foreach (var sname in SOLVER_NAMES)
    {
        var sw = Stopwatch.StartNew();
        var (ok, ms, calls) = RunByName(sname, grid);
        sw.Stop();
        bool timeout = sw.Elapsed.TotalSeconds > TIMEOUT_S;
        bool succes = ok && !timeout;
        Console.WriteLine($"{diffName,6} | {sname,-22} | {succes,-7} | {ms,10:F2} | {calls,10}{(timeout?" [TIMEOUT]":"")}");
    }
}


  Diff | Solveur                | Succes  |  Temps(ms) |     Appels


-----------------------------------------------------------------


  Easy | Backtracking simple    | True    |       1,39 |       4209


  Easy | Backtracking + MRV     | True    |       0,51 |         52


  Easy | AC-3 + backtracking    | True    |       1,22 |         82


...

Medium | DLX (Knuth)            | True    |       0,08 |          0


  Hard | Backtracking simple    | True    |     360,80 |     879418


  Hard | Backtracking + MRV     | True    |     121,86 |       5565


  Hard | AC-3 + backtracking    | True    |    1659,81 |     981113


  Hard | DLX (Knuth)            | True    |       0,31 |          0


---
## Synthese -- ce que disent les chiffres

Les faits marquants (a confronter a la sortie ci-dessus) :

1. **Le backtracking naif explose sur les grilles difficiles** : le facteur de branchement 9 (essai de 1..9 dans l'ordre) sans aucune coupe produit un arbre gigantesque (879 418 appels sur Hard).
2. **MRV divise drastiquement le nombre d'appels** : en choisissant la case la plus contrainte, on remplit les singletons immediatement et on coupe les branches mortes tres tot (Hard passe de 879 418 a 5 565 appels).
3. **AC-3 ajoute une propagation pre-recherche** : la consistance d'arc reduit les domaines avant le backtracking. Sur Easy/Medium le nombre d'appels est faible, mais sur Hard la propagation ne suffit pas a eviter l'explosion (981 113 appels).
4. **DLX est dans une autre categorie** : la formulation exact-cover + Dancing Links visite un nombre de nœuds independant des heuristiques de choix de variable, et chaque operation est O(1). Sur Hard, DLX resout en une fraction de milliseconde la ou le naif met des centaines.

### Parite Python / C# (critere = nombre d'appels recursifs)

Les trois solveurs deterministes (naif, MRV, AC-3) visitent **exactement le meme nombre de nœuds** en C# et en Python :

| Grille | Naif | MRV | AC-3 |
|--------|------|-----|------|
| Easy   | 4209 | 52  | 82   |
| Medium | 55   | 46  | 82   |
| Hard   | 879418 | 5565 | 981113 |

Ces comptes concordent au **nombre pres** entre les deux langages : c'est le critere de parite (algorithme deterministe).

### Ecart runtime : C# .NET vs CPython (G.1 cross-langage, documente honnetement)

Le **temps** wall-clock varie avec le runtime, et cet ecart a un effet visible sur le benchmark :

- **Hard, AC-3** : le Python original **timeout** (75 s > budget 60 s, `succes=False`) apres 981 113 appels. Ce twin C# **termine sous le budget** (~1,7 s) avec le **meme nombre d'appels** (981 113). La difference de succes apparente (False cote Python, True cote C#) n'est **pas** un bug : elle reflete l'avantage du JIT .NET sur CPython pour les boucles tight. Le critere de parite (nombre d'appels) est respecte ; le temps, lui, est runtime-dependant.
- De meme, le naif sur Hard met ~12 s en Python et ~0,3 s en C# (facteur ~35), meme arbre de recherche.

Le point pedagogique (l'explosion combinatoire du naif, le gain de MRV, la superieur asymptotique de DLX) est **identique** dans les deux langages ; seul le mur de temps (le budget de 60 s) est franchi ou non selon le runtime.


---
## 5. Exercices

### Exercice 1 -- Propagation unitaire (unit propagation)

Avant le backtracking, on peut propager les **singletons** : toute case vide dont une seule valeur est possible doit etre remplie immediatement, et cela peut creer de nouveaux singletons (propagation en chaine). Implementez `UnitPropagation(grid)` qui remplit iterativement ces cases et retourne le nombre de remplissages effectues. Comparez ensuite le nombre d'appels de `RunBacktracking` avant et apres propagation unitaire sur la grille Hard.


In [8]:
// Exercice 1 : propagation unitaire. Retourne le nombre de cases remplies, ou -1 si contradiction.
static int UnitPropagation(int[,] grid)
{
    // TODO etudiant : boucler tant qu'on trouve une case vide avec exactement 1 valeur possible.
    // Indice : reutiliser la logique de calcul des valeurs possibles de FindEmptyMRV.
    // Etape 1 : pour chaque case vide, calculer les valeurs possibles.
    // Etape 2 : si une case a exactement 1 valeur possible, la remplir et recommencer.
    // Etape 3 : si une case a 0 valeur possible, retourner -1 (contradiction).
    Console.WriteLine("Exercice a completer -- UnitPropagation");
    return 0;
}

Console.WriteLine("Exercice 1 : methode UnitPropagation definie (stub). A implementer.");


Exercice 1 : methode UnitPropagation definie (stub). A implementer.


### Exercice 2 -- Compteur de nœuds explores pour DLX

Le solveur DLX ci-dessus retourne `calls=0` (il n'instrumente pas le nombre de nœuds visitees). Ajoutez un compteur dans `DLXSolver.Search` (incrementer a chaque entree de `Search`) et retournz-le via `RunDLX`. Comparez ce compte aux `calls` du backtracking naif et de MRV sur la grille Hard : l'ecart quantifie l'avantage de la formulation exact-cover.


In [9]:
// Exercice 2 : instrumenter DLXSolver.Search avec un compteur de nœuds.
// Indice : ajouter un champ 'public int NodesVisited;' dans DLXSolver, l'incrementer en tete de Search.
// Etape 1 : modifier la signature de Search pour accepter/incrementer le compteur.
// Etape 2 : exposer le compteur via une propriete publique.
// Etape 3 : RunDLX retourne (ok, ms, nodesVisited) au lieu de (ok, ms, 0).
Console.WriteLine("Exercice a completer -- instrumentation DLX (voir commentaire ci-dessus).");


Exercice a completer -- instrumentation DLX (voir commentaire ci-dessus).


### Exercice 3 -- Generation de grilles de difficulte controlee

Ecrivez un generateur qui produit une grille de difficulte parametree : (1) resoudre une grille vide avec un solveur pour obtenir une solution complete, (2) retirer k cases au hasard tout en verifier (par re-solution) que la grille reste a solution unique. Le parametre k controle la difficulte. Indice : utiliser `RunBacktracking` comme oracle d'unicite.


In [10]:
// Exercice 3 : generation de grille a difficulte controlee.
static int[,] GeneratePuzzle(int k, int seed)
{
    // TODO etudiant : (1) generer une solution complete, (2) retirer k cases, (3) verifier unicite.
    // Indice : pour l'unicite, resoudre 2 fois avec ordre de parcours different et comparer.
    // Etape 1 : remplir une grille vide par backtracking avec tirage aleatoire des valeurs.
    // Etape 2 : retirer k cases (en sauvegardant leurs positions).
    // Etape 3 : verifier que la grille reduite a une solution unique.
    Console.WriteLine("Exercice a completer -- GeneratePuzzle");
    return new int[9,9];
}

Console.WriteLine("Exercice 3 : methode GeneratePuzzle definie (stub). A implementer.");


Exercice 3 : methode GeneratePuzzle definie (stub). A implementer.


---
## Conclusion

Ce twin C# a reconstruit depuis zero les quatre solveurs Sudoku du notebook Python :

1. **Backtracking naif** -- baseline, explosion combinatoire ;
2. **Backtracking + MRV** -- heuristique de choix de variable, coupe massive ;
3. **AC-3 + backtracking** -- propagation de consistance d'arc avant recherche ;
4. **Dancing Links (Knuth 2000)** -- formulation exact-cover, asymptotiquement superieur.

### Parite Python / C#

- Les **trois grilles** (Easy/Medium/Hard) sont strictement identiques ;
- Le **nombre d'appels recursifs** (naif, MRV, AC-3) concorde exactement entre les deux langages (tableau §4) : c'est le critere de parite (algorithme deterministe) ;
- Le **temps** wall-clock varie avec le runtime (C# .NET JIT ~35x plus rapide que CPython sur le naif Hard), ce n'est pas un critere de parite ;
- **Ecart runtime documente** : sur Hard+AC-3, le Python original timeout (75 s > 60 s) alors que ce twin C# termine (~1,7 s) avec le meme nombre d'appels (981 113). La difference de succes apparente est un effet purement runtime, pas un bug.

### Ponts inter-series

- **Dancing Links / couverture exacte** : cf. la formalisation DLX dans la serie Search et l'algorithme X de Knuth (cf. App-11 Picross pour une autre application de DLX) ;
- **AC-3 / propagation de contraintes** : cf. [Partie 2 (CSP)](../../../Part2-CSP/README.md) pour la theorie de la consistance d'arc ;
- **App-20 Python** : la version originale avec la stdlib Python donne les memes nombres d'appels.

### Prochaines etapes

- Implementer les exercices (propagation unitaire, instrumentation DLX, generation de grilles) ;
- Comparer DLX au backtracking sur des grilles 16x16 ou 25x25 (ou l'ecart s'elargit) ;
- Revenir au [sommaire Search](../../README.md) pour replacer ce notebook dans le parcours global.

## Navigation

[<< App-20 Python](App-20-SudokuBenchmark-Python.ipynb) | [Index](../../README.md) | [App-7 Wordle >>](App-7-Wordle.ipynb)

*Marathon #4956 -- parite .NET / Python, volet Search / Applications / CSP.*
